# 冻结恢复评分与净优势金额
主指标事先固定为评分与净优势的Spearman；其他指标仅辅助解释。完整944日日历区块重采样，不压缩稀疏资格日，不调门槛。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/recovery_score_information_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
def load(name,artifact):
 spec=plan['sources'][name];p=root/spec['card'];manifest=p/'verification_manifest.json';assert hashlib.sha256(manifest.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(manifest.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(str(p/'qlib_recorder')):
  rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);path=Path(rec.download_artifact(artifact));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];return rec.load_object(artifact)
labels=load('labels','analysis/labels');scores=load('scores','analysis/daily')
def metrics(part):
 v=part[['recovery_probability','small_advantage']].dropna();x=v.recovery_probability;y=v.small_advantage
 if len(v)<20 or x.nunique()<2 or y.nunique()<2:return {k:np.nan for k in plan['metrics']}
 loss=(-y).clip(lower=0)
 return {'spearman_net_advantage':float(spearmanr(x,y).statistic),'pearson_net_advantage':float(x.corr(y)),'spearman_loss_severity':float(spearmanr(x,loss).statistic) if loss.nunique()>1 else np.nan,'auc_positive_advantage':float(roc_auc_score(y.gt(0),x)) if y.gt(0).nunique()==2 else np.nan}
intervals=[];annual=[];joined={}
for cost in plan['costs']:
 base=scores[str(cost)].copy();lab=labels.loc[labels.slippage_bps.eq(cost),['trade_date','small_advantage','label_available_at']];frame=base.merge(lab,on='trade_date',how='left',validate='one_to_one')
 assert len(frame)==944 and frame.trade_date.is_unique
 known=frame.small_advantage.notna();assert frame.loc[known,'alpha_ready'].all();assert (frame.loc[known,'label_available_at']>frame.loc[known,'trade_date']+pd.Timedelta(hours=10,minutes=32)).all()
 joined[str(cost)]=frame;point=metrics(frame);n=len(frame);rng=np.random.default_rng(plan['seed']);draws=[]
 for _ in range(plan['resamples']):
  indexes=np.empty(n,dtype=int);indexes[0]=rng.integers(n)
  for j in range(1,n):indexes[j]=rng.integers(n) if rng.random()<1/plan['mean_block'] else (indexes[j-1]+1)%n
  draws.append(metrics(frame.iloc[indexes]))
 resampled=pd.DataFrame(draws)
 for metric in plan['metrics']:
  low,high=np.nanquantile(resampled[metric],[.025,.975]);intervals.append({'slippage_bps':cost,'metric':metric,'calendar_sessions':n,'scored_sessions':int(known.sum()),'estimate':point[metric],'lower':low,'upper':high,'valid_bootstrap_draws':int(resampled[metric].notna().sum())})
 for year,part in frame.groupby(pd.to_datetime(frame.trade_date).dt.year):annual.append({'slippage_bps':cost,'year':year,'scored_sessions':int(part.small_advantage.notna().sum()),**metrics(part)})
summary={'experiment':card.name,'decision':'not_promoted_frozen_recovery_score_information_failed','route_status':'closed_no_threshold_or_sign_search','primary_metric':'spearman_net_advantage','primary_positive_lower_bound_both_costs':all(r['lower']>0 for r in intervals if r['metric']=='spearman_net_advantage'),'new_fits':0,'new_backtests':0,'threshold_searches':0,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Reused-history same-state local recovery beta0.6 labels. Sparse eligibility retained on full 944-session calendar. No independent OOS or complete-policy alpha claim; no multiple-research-selection correction.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/joined']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_recovery_score_information',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,bootstrap_mean_block=plan['mean_block'],resamples=plan['resamples'])
 rec.save_objects(artifact_path='inputs',plan=plan)
 rec.save_objects(artifact_path='analysis',joined=joined,intervals=pd.DataFrame(intervals),annual=pd.DataFrame(annual),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(intervals).to_csv(card/'score_intervals.csv',index=False);pd.DataFrame(annual).to_csv(card/'annual_score_metrics.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));print(pd.DataFrame(intervals).to_string(index=False));print(pd.DataFrame(annual).to_string(index=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']


# 冻结恢复分数：没有可靠的净优势排序证据

0拟合、0回测、0门槛搜索。固定原恢复模型分数与上一轮实际beta0.6的同状态、同重置、Tiered净优势标签。2/5bp分别227/475个成熟合格观测，保留944个原参考日历位置；不把稀疏资格日拼成连续时间。模型、标签、成本路径及来源哈希均固定。

主指标事先固定为Spearman净优势排序。2bp为0.05949，95%区块区间[-0.07026,0.13886]；5bp为-0.01209，区间[-0.10377,0.07415]。两条成本路径均没有正下界。辅助指标：净优势Pearson分别0.00994/-0.00449；AUC分别0.53565区间[0.46108,0.59573]、0.49015区间[0.43001,0.54084]；损失严重度排序也均跨无信息水平。所有1000次重采样有效，平均区块长度63日，未校正此前大量研究选择。

年度主指标2bp在2023/2025为负、2024/2026为正；5bp在2023/2026为负，2024/2025接近0。不存在稳定同向的年度证据。本卡不从年份或辅助指标中另选成功口径。

这表明不能把旧恢复模块不触发仅归咎于0.825门槛太高。即使暂时忽略该门，当前固定分数也未显示可靠区分净收益金额、盈利方向或下跌损失的能力。结果不证明所有分钟信息无效，只关闭以这个旧分数为基础的门槛、反号或校准救活路线。

决定：not_promoted_frozen_recovery_score_information_failed。原分数路线关闭，不再降低门槛、不翻号、不按损益重分箱、不重训同路线。后续若继续寻找特征，必须有新的经济机制与时点合格的信息增量；可逆改写、去重或换坐标本身不产生新信息。当前用户要求仍是不拟合。

局限：这些是反复使用的历史、参考状态上的局部反事实标签，不是完整策略收益，也不是独立样本外；下一共同目标为事后标签终点。2/5bp的状态和样本集合不同，不能将两者当独立重复试验或直接比较收益均值。胜负界限仅用于描述既有标签，不生成交易阈值。生产release未改，稳定净超额未实现。

[固定指标及区间](score_intervals.csv) · [全部年度](annual_score_metrics.csv)

Recorder `65af326e94e1479286fdac66ae2b3e82`。
